# Expand or shrink labels

This notebook takes exisiting integer label maps and **expands or shrinks all objects by a given radius**, keeping the labels intact.

- expansion: labels grow into the background by up to `radius` pixels; in crowded environments, background pixels are assigned to the **closest** object (Euclidean distance, via ```skimage.segmentation.expand_labels```)
- shrinking: each object is eroded by `radius` pixels, object-by-object, so labels never bleed into each other; shared interfaces of touching objects, internal holes and the image edge all erode (per-object erosion)

NOTE: the two operations are not exact inverses of each other (Euclidean vs. structuring element-based distances).

For rings (bands) around object borders instead of whole modified labels, see ```modify_labels_rings.ipynb```.

## Required Inputs

- a folder containing input masks as integer TIFF files

## Outputs

- a new folder with modified label maps as TIFF files (same filenames as input files). Labels stay the same.

In [ ]:
from enum import StrEnum, auto

from skimage.segmentation import expand_labels

from label_morphology import shrink_labels


class Operation(StrEnum):
    EXPAND = auto()
    SHRINK = auto()

In [ ]:
in_path = '/home/david/Desktop/symbiosis_analysis_guse/'

# subfolder containing input labels
labels_subdirectory = 'segmentation_symbionts'
# where to save results to
out_subdirectory = 'segmentation_symbionts_shrunk'

# what to do: 'expand' or 'shrink'
operation = 'shrink'
# radius of operation (pixels)
radius = 5

In [ ]:
from pathlib import Path
from natsort import natsorted

# check that we selected expand or shrink
if operation not in Operation:
    raise ValueError(f'Undefined operation {operation}, available options: {set(Operation)}')

in_files = natsorted((Path(in_path) / labels_subdirectory).glob('[!.]*.tif'))
in_files

In [ ]:
from skimage.io import imread
from tifffile import imwrite

# make out directory if it doesn't exist already
out_path = Path(in_path) / out_subdirectory
if not out_path.exists():
    out_path.mkdir()

for in_file in in_files:

    # load
    labels = imread(in_file)

    # apply either expansion or shrinking
    if operation == Operation.EXPAND:
        labels_modified = expand_labels(labels, radius)
    else:
        labels_modified = shrink_labels(labels, radius)

    # save to output folder with same name
    out_file = out_path / in_file.name
    imwrite(out_file, labels_modified, compression=5)
    
    print(f'{in_file} done.')

## TESTING code below -> commented out for papermill usage

In [ ]:
# from matplotlib import pyplot as plt
# from skimage.io import imread

# labels = imread('test_labels.tif')
# labels_shrunk = shrink_labels(labels, 5)
# labels_expanded = expand_labels(labels, 5)

# fig, axs = plt.subplots(ncols=3, figsize=(12,4))
# axs[0].imshow(labels)
# axs[1].imshow(labels_shrunk)
# axs[2].imshow(labels_expanded)

In [ ]:
# import napari

# if napari.current_viewer() is not None:
#     napari.current_viewer().close()

# viewer = napari.Viewer()

# viewer.add_labels(labels_modified)